### **Chapter 14 Problems**
Compute using $h=2^{-1}, 2^{-2}, \ldots 2^{-5}$ and the forward, backward, and centered difference the following derivatives.

### **Problem 14.1**
$$
f(x)=\sqrt{x} \text { at } x=0.5 \text {. The answer is } f^{\prime}(0.5)=2^{-1 / 2} \approx 0.70710678118 \text {. }
$$

In [4]:
import numpy as np

def func(x):
    return np.sqrt(x)

x = 0.5

true_deriv = 2**(-0.5) # True derivative is 2^(-1/2) ~ 0.70710678118
hs = [2**-1, 2**-2, 2**-3, 2**-4, 2**-5]

print("h\tforward\t\tbackward\tcentered\ttrue")
for h in hs:
    fwd = (func(x + h) - func(x)) / h
    bwd = (func(x) - func(x - h)) / h
    cen = (func(x + h) - func(x - h)) / (2 * h)
    print(f"{h:.5g}\t{fwd:.12f}\t{bwd:.12f}\t{cen:.12f}\t{true_deriv:.12f}")


h	forward		backward	centered	true
0.5	0.585786437627	1.414213562373	1.000000000000	0.707106781187
0.25	0.635674490392	0.828427124746	0.732050807569	0.707106781187
0.125	0.667701070844	0.757874763926	0.712787917385	0.707106781187
0.0625	0.686291501015	0.730703254726	0.708497377871	0.707106781187
0.03125	0.696390581412	0.718514697763	0.707452639587	0.707106781187


### **Problem 14.2**
$$
f(x)=\arctan \left(x^2-0.9 x+2\right) \text { at } x=0.5 . \text { The answer is } f^{\prime}(0.5)=\frac{5}{212} .
$$

In [5]:
import numpy as np

def func(x):
    return np.arctan(x**2 - 0.9*x + 2)

x = 0.5

true_deriv = 5/212 # True derivative is 5/212
hs = [2**-1, 2**-2, 2**-3, 2**-4, 2**-5]

print("h\tforward\t\tbackward\tcentered\ttrue")
for h in hs:
    fwd = (func(x + h) - func(x)) / h
    bwd = (func(x) - func(x - h)) / h
    cen = (func(x + h) - func(x - h)) / (2 * h)
    print(f"{h:.5g}\t{fwd:.12f}\t{bwd:.12f}\t{cen:.12f}\t{true_deriv:.12f}")


h	forward		backward	centered	true
0.5	0.125358588982	-0.086901790783	0.019228399100	0.023584905660
0.25	0.079580175242	-0.034822103464	0.022379035889	0.023584905660
0.125	0.052439161542	-0.005888413481	0.023275374031	0.023584905660
0.0625	0.038160864051	0.008853147497	0.023507005774	0.023584905660
0.03125	0.030901372680	0.016229423663	0.023565398171	0.023584905660


### **Chapter 15 Problems**
Using the trapezoid rule and Simpson’s rule estimate the following integrals with the following number of intervals: 2, 4, 8, 16,... 512. Compare your answers with Romberg integration where the maximum number of levels set to 9.

In [6]:
import numpy as np
def trapezoid(f, a, b, pieces):
    """Find the integral of the function f between a and b
    using pieces trapezoids
    Args:
        f: function to integrate
        a: lower bound of integral
        b: upper bound of integral
        pieces: number of pieces to chop [a,b] into
    Returns:
        estimate of integral
    """
    integral = 0
    h = b - a
    #initialize the left function evaluation
    fa = f(a)
    for i in range(pieces):
        #evaluate the function at the left end of the piece
        fb = f(a+(i+1)*h/pieces)
        integral += 0.5*h/pieces*(fa + fb)
        #now make the left function evaluation the right for the next step
        fa = fb
    return integral

def simpsons(f, a, b, pieces):
    """Find the integral of the function f between a and b
    using Simpson’s rule
    Args:
        f: function to integrate
        a: lower bound of integral
        b: upper bound of integral
        pieces: number of pieces to chop [a,b] into
    Returns:
        estimate of integral
    """
    integral = 0
    h = b - a
    one_sixth = 1.0/6.0
    #initialize the left function evaluation
    fa = f(a)
    for i in range(pieces):
        #evaluate the function at the left end of the piece
        fb = f(a+(i+1)*h/pieces)
        fmid = f(0.5*(a+(i+1)*h/pieces+ a+i*h/pieces))
        integral += one_sixth*h/pieces*(fa + 4*fmid + fb)
        #now make the left function evaluation the right for the next step
        fa = fb
    return integral

import decimal
#set precision to be 100 digits

def RichardsonExtrapolation(fh, fhn, n, k):
    """Compute the Richardson extrapolation based on two approximations of order k
    where the finite difference parameter h is used in fh and h/n in fhn.
    Inputs:
    fh:  Approximation using h
    fhn: Approximation using h/n
    n:   divisor of h
    k:   original order of approximation
    
    Returns:
    Richardson estimate of order k+1"""

    ''' Decimal is a number type that provides:
	 - Arbitrary precision
	 - Exact base-10 arithmetic
	 - Better control over rounding
    '''
    n = decimal.Decimal(str(n))
    k = decimal.Decimal(str(k))
    numerator = decimal.Decimal(n**k * decimal.Decimal(fhn) - decimal.Decimal(fh))
    denominator = decimal.Decimal(n**k - decimal.Decimal(1.0))
    return float(numerator/denominator)

def Romberg(f, a, b, MaxLevels = 10, epsilon = 1.0e-6, PrintMatrix = False):
    """Compute the Romberg integral of f from a to b
    Inputs:
    f:  integrand function
    a: left edge of integral
    b: right edge of integral
    MaxLevels: Number of levels to take the integration to
    
    Returns:
    Romberg integral estimate"""
    
    estimate = np.zeros((MaxLevels,MaxLevels))
    
    estimate[0,0] = trapezoid(f,a,b,pieces=1)
    count = 1
    converged = 0
    while not(converged):
        estimate[count,0] = trapezoid(f,a,b,pieces=2**count)
        for extrap in range(count):
            estimate[count,1+extrap] = RichardsonExtrapolation(estimate[count-1,extrap],
                                                               estimate[count,extrap],2,2**(extrap+1))
        
        converged = np.fabs(estimate[count,count] - estimate[count-1,count-1]) < epsilon
        if (count == MaxLevels-1): converged = 1
        count += 1
    if (PrintMatrix):
        print(estimate[0:count,0:count])
    return estimate[count-1, count-1]

### **Problem 15.1**
$$
\int_0^{\pi / 2} e^{\sin x} d x \approx 3.104379017855555098181
$$

In [11]:
intervals = [2, 4, 8, 16, 32, 64, 128, 256, 512]

lower_bound = 0
upper_bound = np.pi / 2
def func2(x):
    return np.exp(np.sin(x))


print("Intervals\tTrapezoid\tSimpson\t\tRomberg")

for i in intervals:
    trap = trapezoid(func2, lower_bound, upper_bound, i)
    simp = simpsons(func2, lower_bound, upper_bound, i)
    romb = Romberg(func2, lower_bound, upper_bound, MaxLevels=9)
    print(f"{i:.5g}\t\t{trap:.12f}\t{simp:.12f}\t{romb:.12f}")

Intervals	Trapezoid	Simpson		Romberg
2		3.053043641278	3.104357408428	3.104379029273
4		3.091528966640	3.104378706144	3.104379029273
8		3.101166271268	3.104379013085	3.104379029273
16		3.103575827630	3.104379017781	3.104379029273
32		3.104178220244	3.104379017854	3.104379029273
64		3.104328818452	3.104379017856	3.104379029273
128		3.104366468005	3.104379017856	3.104379029273
256		3.104375880393	3.104379017856	3.104379029273
512		3.104378233490	3.104379017856	3.104379029273


### **Problem 15.1**
$\int_0^{2.405} J_0(x) d x \approx 1.470300035485$, where $J_0(x)$ is a Bessel function of the first kind given by
$$
J_\alpha(x)=\sum_{m=0}^{\infty} \frac{(-1)^m}{m!\Gamma(m+\alpha+1)}\left(\frac{x}{2}\right)^{2 m+\alpha} .
$$

In [16]:
from scipy.special import j0

intervals = [2, 4, 8, 16, 32, 64, 128, 256, 512]

lower_bound = 0
upper_bound = 2.405
def func2(x):
    return j0(x)

print("Intervals\tTrapezoid\tSimpson\t\tRomberg")

for i in intervals:
    trap = trapezoid(func2, lower_bound, upper_bound, i)
    simp = simpsons(func2, lower_bound, upper_bound, i)
    romb = Romberg(func2, lower_bound, upper_bound, MaxLevels=9)
    print(f"{i:.5g}\t\t{trap:.12f}\t{simp:.12f}\t{romb:.12f}")

Intervals	Trapezoid	Simpson		Romberg
2		1.406733735052	1.470555068807	1.470300030653
4		1.454599735368	1.470315573631	1.470300030653
8		1.466386614065	1.470301000534	1.470300030653
16		1.469322403917	1.470300095707	1.470300030653
32		1.470055672759	1.470300039248	1.470300030653
64		1.470238947626	1.470300035721	1.470300030653
128		1.470284763697	1.470300035500	1.470300030653
256		1.470296217549	1.470300035486	1.470300030653
512		1.470299081002	1.470300035486	1.470300030653
